# NB_patient_360

Builds `caresyncadb.gold.patient_360` from the silver layer per the STTM (Silver to Gold v1.0). Grain: one row per patient. Driving table `silver.patients` with LEFT JOINs to hospitals, insurance_providers and aggregated lab_results. Load: full rebuild (CREATE OR REPLACE TABLE AS SELECT).

In [0]:
%sql
-- Full rebuild: one row per patient (driving table: silver.patients)
CREATE OR REPLACE TABLE caresyncadb.gold.patient_360 AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                      AS total_tests,
    COUNT_IF(result_status = 'Abnormal')          AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')          AS critical_test_count,
    MAX(test_date)                                AS last_test_date
  FROM caresyncadb.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                          AS date_of_birth,
  CAST(FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  ip.provider_name                                                       AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                        AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                        AS critical_test_count,
  l.last_test_date
FROM caresyncadb.silver.patients p
LEFT JOIN caresyncadb.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN caresyncadb.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id

num_affected_rows,num_inserted_rows


In [0]:
# Read the rebuilt gold table and return its record count to the caller (e.g. a job or dbutils.notebook.run)
updated_df = spark.table("caresyncadb.gold.patient_360")
record_count = updated_df.count()
print(f"caresyncadb.gold.patient_360 record count: {record_count}")
dbutils.notebook.exit(str(record_count))